In [31]:
import polars as pl
import numpy as np
from sklearn.cluster import KMeans
import pandas as pd
import os


### **Đặc trưng:** Phân khúc loại sản phẩm (category_l1) theo mức giá: Bình dân - Trung cấp - Cao cấp

In [18]:
path_item = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.item_chunk_0.parquet"

df_item = pl.read_parquet(path_item)
print(df_item.shape)
print(df_item.head())


(27323, 11)
shape: (5, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ item_id   ┆ price     ┆ category_ ┆ category_ ┆ … ┆ gender_ta ┆ descripti ┆ brand_fin ┆ age_grou │
│ ---       ┆ ---       ┆ l1        ┆ l2        ┆   ┆ rget_fina ┆ on_final  ┆ al        ┆ p_final  │
│ str       ┆ decimal[3 ┆ ---       ┆ ---       ┆   ┆ l         ┆ ---       ┆ ---       ┆ ---      │
│           ┆ 8,4]      ┆ str       ┆ str       ┆   ┆ ---       ┆ str       ┆ str       ┆ str      │
│           ┆           ┆           ┆           ┆   ┆ str       ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 050202000 ┆ 99000.000 ┆ Babycare  ┆ Bình sữa, ┆ … ┆ Không xác ┆ Chi tiết  ┆ Dr.Brown' ┆ Từ 9M    │
│ 0004      ┆ 0         ┆           ┆ phụ kiện  ┆   ┆ định      ┆ sản phẩm  ┆ s         ┆          │
│           ┆           ┆           ┆           ┆   ┆           

In [19]:
def cluster_price_group(prices):
    prices_reshaped = prices.to_numpy().reshape(-1, 1)

    # Gom cụm 3 cluster
    kmeans = KMeans(n_clusters=3, random_state=42)
    labels = kmeans.fit_predict(prices_reshaped)

    # Tính mean của từng cluster bằng pandas
    df_temp = pd.DataFrame({"price": prices.values, "cluster": labels})
    cluster_means = df_temp.groupby("cluster")["price"].mean()

    # Sắp xếp cluster theo giá tăng dần → 0,1,2
    sorted_clusters = cluster_means.sort_values().index.tolist()

    # Mapping cluster gốc → 0,1,2
    cluster_map = {sorted_clusters[i]: i for i in range(3)}

    # Trả về danh sách nhãn theo thứ tự index
    return [cluster_map[c] for c in labels]

In [20]:
pdf = df_item.select(["item_id", "price", "category_l1"]).to_pandas()

# Tạo cột rỗng
pdf["price_segment"] = -1

for cat, group in pdf.groupby("category_l1"):
    X = group["price"].values.reshape(-1,1)

    if len(group) < 3:
        # Không gom cụm nếu số lượng ít → gán 0 hết
        pdf.loc[group.index, "price_segment"] = 0
        continue

    # Gom cụm
    kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
    labels = kmeans.fit_predict(X)

    # Remap theo thứ tự giá trung bình
    cluster_mean = pd.DataFrame({
        "cluster": labels,
        "price": group["price"].values
    }).groupby("cluster")["price"].mean().sort_values()

    mapping = {cluster: rank for rank, cluster in enumerate(cluster_mean.index)}

    # Gán nhãn theo index
    pdf.loc[group.index, "price_segment"] = [mapping[c] for c in labels]


In [21]:
print(pdf.groupby(["category_l1", "price_segment"]).size())


category_l1             price_segment
Babycare                0                1795
                        1                 165
                        2                  34
Gói Hội Viên            0                   3
                        1                   1
                        2                   3
Hóa mỹ phẩm cho bé      0                 168
                        1                 148
                        2                  29
Hóa mỹ phẩm gia đình    0                 178
                        1                 183
                        2                  26
Phụ kiện                0                1492
                        1                1134
                        2                 521
Sữa                     0                 159
                        1                 205
                        2                  73
Sữa nước                0                 131
                        1                  27
                        2                 

Mapping:
- 0: Bình dân
- 1: Trung cấp
- 2: Cao cấp

In [22]:
segment_stats = (
    pdf.groupby(["category_l1", "price_segment"])["price"]
       .agg(["min", "max", "mean", "median", "count"])
       .sort_values(["category_l1", "price_segment"])
)

print("\n=== BẢNG THỐNG KÊ THEO SEGMENT ===")
print(segment_stats)



=== BẢNG THỐNG KÊ THEO SEGMENT ===
                                               min            max  \
category_l1            price_segment                                
Babycare               0                 1000.0000   1767273.0000   
                       1              1821000.0000   6775000.0000   
                       2              6950000.0000  20990000.0000   
Gói Hội Viên           0                19000.0000     99000.0000   
                       1               169000.0000    169000.0000   
                       2               230000.0000    299000.0000   
Hóa mỹ phẩm cho bé     0                20000.0000    140000.0000   
                       1               145000.0000    290000.0000   
                       2               295000.0000    685000.0000   
Hóa mỹ phẩm gia đình   0                12000.0000    135000.0000   
                       1               139000.0000    275000.0000   
                       2               288000.0000    750000.0000  

In [26]:
df_price_seg = pl.from_pandas(pdf)


In [27]:
df_item = df_item.join(
    df_price_seg.select(["item_id", "price_segment"]),
    on="item_id",
    how="left"
)


In [28]:
print(df_item.head())
print(df_item.select("price_segment").unique())


shape: (5, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ item_id   ┆ price     ┆ category_ ┆ category_ ┆ … ┆ descripti ┆ brand_fin ┆ age_group ┆ price_se │
│ ---       ┆ ---       ┆ l1        ┆ l2        ┆   ┆ on_final  ┆ al        ┆ _final    ┆ gment    │
│ str       ┆ decimal[3 ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ 8,4]      ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ i64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 050202000 ┆ 99000.000 ┆ Babycare  ┆ Bình sữa, ┆ … ┆ Chi tiết  ┆ Dr.Brown' ┆ Từ 9M     ┆ 0        │
│ 0004      ┆ 0         ┆           ┆ phụ kiện  ┆   ┆ sản phẩm  ┆ s         ┆           ┆          │
│           ┆           ┆           ┆           ┆   ┆ …         ┆           ┆           ┆          │
│ 001029004 ┆ 69000.000 ┆ Thời      ┆ Cơ cấu    ┆ … ┆ Không xác ┆ Con Cưng  

In [30]:
output_path = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\sale_pers.item_chunk_0.parquet"

df_item.write_parquet(output_path)

print(f"Đã lưu thành công vào:\n{output_path}")


Đã lưu thành công vào:
D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\sale_pers.item_chunk_0.parquet


**Nhận Xét:**

**1. Các nhóm ngành có phân khúc giá rất rõ ràng và hợp lý**

Những category_l1 như Sữa, Tã, Textile, Đồ chơi & Sách, TPCN, Hóa mỹ phẩm, Babycare thể hiện phân tách giá rất mạnh:
- Cụm 0 (Bình dân): giá thấp – trung bình
- Cụm 1 (Trung cấp): giá tăng đáng kể
- Cụm 2 (Cao cấp): giá cao vượt trội và rất đặc trưng

**2. Các ngành hàng giá thấp như “Phụ kiện”, “Thực phẩm cho bé”, “Vệ sinh” không quá chênh lệch giữa 0 - 1 - 2**

=> Phân bố không quá rõ ràng để ứng dụng phân cụm

**3. Chỉ có "Tã" và "Sữa" là có phân khúc Trung cấp nhiều hơn Cao cấp**

In [ ]:
#===================================

### **Đặc trưng:** Số loại sản phẩm (category_l1) trung bình cho một lần mua hàng (một ngày), và phân cụm số loại giao dịch trung bình đó, gán vào 3 nhãn "Mua ít", "Mua vừa" và "Mua nhiều".

In [23]:
path_tx_0 = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_0.parquet"

df_tx0 = pl.read_parquet(path_tx_0)

print("Số dòng:", df_tx0.height)
print("Số cột:", df_tx0.width)
print("\nDanh sách cột:")
print(df_tx0.columns)


Số dòng: 1786491
Số cột: 12

Danh sách cột:
['item_id', 'price', 'quantity', 'customer_id', 'created_date', 'channel', 'payment', 'location', 'discount', 'list_price', 'category_l2', 'discount_rate']


In [24]:
df_tx0.head()

item_id,price,quantity,customer_id,created_date,channel,payment,location,discount,list_price,category_l2,discount_rate
str,"decimal[38,4]",i32,i32,date,str,str,i32,"decimal[38,4]","decimal[38,4]",str,"decimal[38,4]"
"""7115000000004""",49000.0000,1,5254214,2024-12-24,"""In-Store""","""VietQR""",656,0.0000,49000.0000,"""Snack ăn dặm""",0.0000
"""0029130000030""",69000.0000,1,7573232,2024-12-24,"""In-Store""","""Tiền mặt""",143,0.0000,74000.0000,"""Bột ăn dặm""",0.0676
"""3496000000053""",75000.0000,2,8187418,2024-12-24,"""In-Store""","""MoMo""",213,0.0000,75000.0000,"""Quần áo & Phụ kiện sơ sinh""",0.0000
"""2700000000002""",58500.0000,2,8187418,2024-12-24,"""In-Store""","""MoMo""",213,13000.0000,65000.0000,"""Khăn khô""",0.1000
"""0029110000036""",89000.0000,1,6931560,2024-12-28,"""Android""","""MoMo""",590,10000.0000,99000.0000,"""Snack ăn dặm""",0.1010


In [33]:
# Thư mục chứa các file transaction 0..19
TX_DIR = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data"
TX_PATTERN = "sale_pers.purchase_history_daily_chunk_{}.parquet"

# File item đã preprocessing / feature engineering
ITEM_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\sale_pers.item_chunk_0.parquet"


In [34]:
# Đọc item
df_item = pl.read_parquet(ITEM_PATH)
print("Item shape:", df_item.shape)

# Chỉ giữ các cột cần thiết cho đặc trưng này
df_item_small = df_item.select(["item_id", "category_l1"])
print(df_item_small.head())


Item shape: (27323, 12)
shape: (5, 2)
┌───────────────┬────────────────┐
│ item_id       ┆ category_l1    │
│ ---           ┆ ---            │
│ str           ┆ str            │
╞═══════════════╪════════════════╡
│ 0502020000004 ┆ Babycare       │
│ 0010290040150 ┆ Thời trang     │
│ 0008010000015 ┆ Đồ chơi & Sách │
│ 0020010000094 ┆ Tã             │
│ 0020010000098 ┆ Tã             │
└───────────────┴────────────────┘


In [35]:
customer_agg_chunks = []  # list để lưu aggregate theo chunk

for i in range(20):  # từ 0 đến 19
    tx_path = os.path.join(TX_DIR, TX_PATTERN.format(i))
    print(f"\n=== Đang xử lý chunk {i}: {tx_path} ===")
    
    # Đọc tối thiểu 3 cột
    tx_chunk = pl.read_parquet(
        tx_path,
        columns=["item_id", "customer_id", "created_date"]
    )
    print("  Transaction chunk shape:", tx_chunk.shape)
    
    # Join với item để lấy category_l1
    tx_joined = tx_chunk.join(
        df_item_small,
        on="item_id",
        how="left"
    )
    
    # Group theo (customer_id, created_date) để đếm số loại category_l1 / ngày
    daily_cat_count = (
        tx_joined
        .group_by(["customer_id", "created_date"])
        .agg(
            pl.col("category_l1").n_unique().alias("unique_categories")
        )
    )
    
    print("  daily_cat_count shape:", daily_cat_count.shape)
    
    # Aggregate theo customer_id trong từng chunk:
    # - tổng số loại
    # - số ngày mua hàng
    cust_chunk = (
        daily_cat_count
        .group_by("customer_id")
        .agg([
            pl.col("unique_categories").sum().alias("sum_unique_categories"),
            pl.len().alias("num_days")
        ])
    )
    
    print("  cust_chunk shape:", cust_chunk.shape)
    
    customer_agg_chunks.append(cust_chunk)

print("\n=== Hoàn thành xử lý tất cả chunk transaction ===")



=== Đang xử lý chunk 0: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_0.parquet ===
  Transaction chunk shape: (1786491, 3)
  daily_cat_count shape: (768604, 3)
  cust_chunk shape: (486150, 3)

=== Đang xử lý chunk 1: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_1.parquet ===
  Transaction chunk shape: (1786491, 3)
  daily_cat_count shape: (732049, 3)
  cust_chunk shape: (464722, 3)

=== Đang xử lý chunk 2: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_2.parquet ===
  Transaction chunk shape: (1786491, 3)
  daily_cat_count shape: (741831, 3)
  cust_chunk shape: (482378, 3)

=== Đang xử lý chunk 3: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_3.parquet ===
  Tra

In [36]:
# Gộp tất cả aggregate từng chunk
customer_agg_all = pl.concat(customer_agg_chunks, how="vertical")
print("customer_agg_all shape:", customer_agg_all.shape)

# Group lần cuối theo customer_id để cộng dồn các chunk
customer_final = (
    customer_agg_all
    .group_by("customer_id")
    .agg([
        pl.col("sum_unique_categories").sum().alias("total_unique_categories"),
        pl.col("num_days").sum().alias("total_days")
    ])
)

# Tính trung bình số loại category_l1 mỗi ngày
customer_final = customer_final.with_columns(
    (pl.col("total_unique_categories") / pl.col("total_days")).alias("avg_categories_per_day")
)

print("\n=== Bảng đặc trưng trung bình số loại category_l1 / ngày ===")
print(customer_final.head())
print("Số lượng khách hàng:", customer_final.height)


customer_agg_all shape: (9875793, 3)

=== Bảng đặc trưng trung bình số loại category_l1 / ngày ===
shape: (5, 4)
┌─────────────┬─────────────────────────┬────────────┬────────────────────────┐
│ customer_id ┆ total_unique_categories ┆ total_days ┆ avg_categories_per_day │
│ ---         ┆ ---                     ┆ ---        ┆ ---                    │
│ i32         ┆ u32                     ┆ u32        ┆ f64                    │
╞═════════════╪═════════════════════════╪════════════╪════════════════════════╡
│ 5919834     ┆ 26                      ┆ 19         ┆ 1.368421               │
│ 8018318     ┆ 2                       ┆ 1          ┆ 2.0                    │
│ 6476953     ┆ 32                      ┆ 24         ┆ 1.333333               │
│ 6608138     ┆ 23                      ┆ 20         ┆ 1.15                   │
│ 6289095     ┆ 10                      ┆ 10         ┆ 1.0                    │
└─────────────┴─────────────────────────┴────────────┴────────────────────────┘
Số lượn

In [38]:
customer_final = customer_final.with_columns(
    (pl.col("total_unique_categories") / pl.col("total_days")).alias("avg_categories_per_day")
)

print("\n=== 10 khách hàng ngẫu nhiên để kiểm tra kết quả trung gian ===")

sample_10 = (
    customer_final
    .select(["customer_id", "total_unique_categories", "total_days", "avg_categories_per_day"])
    .sample(n=10, shuffle=True)
)

print(sample_10)




=== 10 khách hàng ngẫu nhiên để kiểm tra kết quả trung gian ===
shape: (10, 4)
┌─────────────┬─────────────────────────┬────────────┬────────────────────────┐
│ customer_id ┆ total_unique_categories ┆ total_days ┆ avg_categories_per_day │
│ ---         ┆ ---                     ┆ ---        ┆ ---                    │
│ i32         ┆ u32                     ┆ u32        ┆ f64                    │
╞═════════════╪═════════════════════════╪════════════╪════════════════════════╡
│ 8138368     ┆ 3                       ┆ 2          ┆ 1.5                    │
│ 7461408     ┆ 1                       ┆ 1          ┆ 1.0                    │
│ 7316610     ┆ 23                      ┆ 22         ┆ 1.045455               │
│ 7710563     ┆ 1                       ┆ 1          ┆ 1.0                    │
│ 7134660     ┆ 1                       ┆ 1          ┆ 1.0                    │
│ 7323665     ┆ 1                       ┆ 1          ┆ 1.0                    │
│ 7764747     ┆ 1                       

| Cột                       | Ý nghĩa                                                  |
| ------------------------- | -------------------------------------------------------- |
| `total_unique_categories` | Tổng số loại category_l1 khách mua trong toàn bộ lịch sử |
| `total_days`              | Số ngày khách có mua hàng                                |
| `avg_categories_per_day`  | Số loại category_l1 trung bình mỗi ngày khách mua        |


In [39]:
# Lấy dữ liệu để gom cụm
X = customer_final["avg_categories_per_day"].to_numpy().reshape(-1, 1)

print("\nGiá trị min/max của avg_categories_per_day:")
print("  Min:", X.min(), " Max:", X.max())

# KMeans 3 cụm
kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
labels = kmeans.fit_predict(X)

# Thêm nhãn cụm vào bảng
customer_final = customer_final.with_columns(
    pl.Series("buy_segment_raw", labels)
)



Giá trị min/max của avg_categories_per_day:
  Min: 1.0  Max: 11.0


In [44]:
# === Tính mean avg_categories_per_day theo cụm ===
cluster_stats = (
    customer_final
    .group_by("buy_segment_raw")
    .agg(
        pl.col("avg_categories_per_day").mean().alias("mean_avg_categories")
    )
    .sort("mean_avg_categories")
)

print("\n=== Mean avg_categories_per_day theo cụm KMeans (trước remap) ===")
print(cluster_stats)

# === Tạo mapping từ cluster gốc -> cluster mới (0: ít, 1: vừa, 2: nhiều) ===
mapping = {}
for new_label, row in enumerate(cluster_stats.iter_rows(named=True)):
    orig_cluster = row["buy_segment_raw"]
    mapping[orig_cluster] = new_label

print("\nMapping cụm:")
for orig, new in mapping.items():
    print(f"  Cluster {orig} -> {new}")

# === Áp dụng mapping ===
customer_final = customer_final.with_columns(
    pl.col("buy_segment_raw").replace(mapping).alias("buy_segment")
)

print("\n=== Kết quả cuối cùng (mẫu) ===")
print(customer_final.select(["customer_id", "avg_categories_per_day", "buy_segment"]).head(10))



=== Mean avg_categories_per_day theo cụm KMeans (trước remap) ===
shape: (3, 2)
┌─────────────────┬─────────────────────┐
│ buy_segment_raw ┆ mean_avg_categories │
│ ---             ┆ ---                 │
│ i32             ┆ f64                 │
╞═════════════════╪═════════════════════╡
│ 0               ┆ 1.070506            │
│ 1               ┆ 1.848658            │
│ 2               ┆ 3.419495            │
└─────────────────┴─────────────────────┘

Mapping cụm:
  Cluster 0 -> 0
  Cluster 1 -> 1
  Cluster 2 -> 2

=== Kết quả cuối cùng (mẫu) ===
shape: (10, 3)
┌─────────────┬────────────────────────┬─────────────┐
│ customer_id ┆ avg_categories_per_day ┆ buy_segment │
│ ---         ┆ ---                    ┆ ---         │
│ i32         ┆ f64                    ┆ i32         │
╞═════════════╪════════════════════════╪═════════════╡
│ 5919834     ┆ 1.368421               ┆ 0           │
│ 8018318     ┆ 2.0                    ┆ 1           │
│ 6476953     ┆ 1.333333               ┆ 0 

In [46]:
segment_check = (
    customer_final
    .group_by("buy_segment")
    .agg([
        pl.count().alias("num_customers"),
        pl.col("avg_categories_per_day").min().alias("min_avg"),
        pl.col("avg_categories_per_day").median().alias("median_avg"),
        pl.col("avg_categories_per_day").mean().alias("mean_avg"),
        pl.col("avg_categories_per_day").max().alias("max_avg"),
    ])
    .sort("buy_segment")
)

print("\n=== Thống kê theo từng segment mua (ít = 0 / vừa = 1 / nhiều = 2) ===")
print(segment_check)



=== Thống kê theo từng segment mua (ít = 0 / vừa = 1 / nhiều = 2) ===
shape: (3, 6)
┌─────────────┬───────────────┬──────────┬────────────┬──────────┬──────────┐
│ buy_segment ┆ num_customers ┆ min_avg  ┆ median_avg ┆ mean_avg ┆ max_avg  │
│ ---         ┆ ---           ┆ ---      ┆ ---        ┆ ---      ┆ ---      │
│ i32         ┆ u32           ┆ f64      ┆ f64        ┆ f64      ┆ f64      │
╞═════════════╪═══════════════╪══════════╪════════════╪══════════╪══════════╡
│ 0           ┆ 1488503       ┆ 1.0      ┆ 1.0        ┆ 1.070506 ┆ 1.459184 │
│ 1           ┆ 818176        ┆ 1.459259 ┆ 1.880952   ┆ 1.848658 ┆ 2.633333 │
│ 2           ┆ 135627        ┆ 2.634146 ┆ 3.0        ┆ 3.419495 ┆ 11.0     │
└─────────────┴───────────────┴──────────┴────────────┴──────────┴──────────┘


C:\Users\PC\AppData\Local\Temp\ipykernel_7772\1373689411.py:5: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("num_customers"),


In [48]:
for seg in [0, 1, 2]:
    print(f"\n--- Ví dụ khách hàng thuộc segment {seg} ---")
    sample = (
        customer_final
        .filter(pl.col("buy_segment") == seg)
        .select(["customer_id", "avg_categories_per_day", "buy_segment"])  
        .head(10)
    )
    print(sample)



--- Ví dụ khách hàng thuộc segment 0 ---
shape: (10, 3)
┌─────────────┬────────────────────────┬─────────────┐
│ customer_id ┆ avg_categories_per_day ┆ buy_segment │
│ ---         ┆ ---                    ┆ ---         │
│ i32         ┆ f64                    ┆ i32         │
╞═════════════╪════════════════════════╪═════════════╡
│ 5919834     ┆ 1.368421               ┆ 0           │
│ 6476953     ┆ 1.333333               ┆ 0           │
│ 6608138     ┆ 1.15                   ┆ 0           │
│ 6289095     ┆ 1.0                    ┆ 0           │
│ 8175224     ┆ 1.0                    ┆ 0           │
│ 2629334     ┆ 1.0                    ┆ 0           │
│ 7418603     ┆ 1.444444               ┆ 0           │
│ 3761233     ┆ 1.153846               ┆ 0           │
│ 6994834     ┆ 1.0                    ┆ 0           │
│ 4626489     ┆ 1.0                    ┆ 0           │
└─────────────┴────────────────────────┴─────────────┘

--- Ví dụ khách hàng thuộc segment 1 ---
shape: (10, 3)
┌─────

In [49]:
segment_counts = (
    customer_final
    .group_by("buy_segment")
    .agg(pl.count().alias("num_customers"))
    .sort("buy_segment")
)

print("=== Số lượng khách hàng theo từng segment (0=ít, 1=vừa, 2=nhiều) ===")
print(segment_counts)


=== Số lượng khách hàng theo từng segment (0=ít, 1=vừa, 2=nhiều) ===
shape: (3, 2)
┌─────────────┬───────────────┐
│ buy_segment ┆ num_customers │
│ ---         ┆ ---           │
│ i32         ┆ u32           │
╞═════════════╪═══════════════╡
│ 0           ┆ 1488503       │
│ 1           ┆ 818176        │
│ 2           ┆ 135627        │
└─────────────┴───────────────┘


C:\Users\PC\AppData\Local\Temp\ipykernel_7772\670758724.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("num_customers"))


In [50]:
# Đặt đường dẫn xuất file
output_path = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\customer_behavior.parquet"

# Tạo thư mục nếu chưa tồn tại
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Lưu dataframe
customer_final.write_parquet(output_path)

print("Đã lưu file thành công tại:")
print(output_path)

Đã lưu file thành công tại:
D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\customer_behavior.parquet


**Biến chính là buy_segment** còn lại là biến trung gian

### **Đặc trưng:** Mức độ cao cấp của khách hàng - dựa trên "Sữa", "Tã"
- Bước 1: Gom nhóm giá các mặt hàng có category_l1 = 'sữa' theo các nhãn "Bình dân", "Trung cấp" và "cao cấp".

- Bước 2: Thống kê xem khách hàng đó mua sữa thuộc nhóm nào nhiều nhất để gán nhãn cho khách hàng đó.

In [51]:

# Đường dẫn item file đã preprocessing
path_item = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\sale_pers.item_chunk_0.parquet"

# Chỉ load cột cần thiết
df_item = pl.read_parquet(
    path_item,
    columns=["item_id", "category_l1", "price_segment"]
)

print("ITEM shape:", df_item.shape)
print(df_item.head())


ITEM shape: (27323, 3)
shape: (5, 3)
┌───────────────┬────────────────┬───────────────┐
│ item_id       ┆ category_l1    ┆ price_segment │
│ ---           ┆ ---            ┆ ---           │
│ str           ┆ str            ┆ i64           │
╞═══════════════╪════════════════╪═══════════════╡
│ 0502020000004 ┆ Babycare       ┆ 0             │
│ 0010290040150 ┆ Thời trang     ┆ 0             │
│ 0008010000015 ┆ Đồ chơi & Sách ┆ 0             │
│ 0020010000094 ┆ Tã             ┆ 1             │
│ 0020010000098 ┆ Tã             ┆ 1             │
└───────────────┴────────────────┴───────────────┘


In [52]:
# Nơi lưu nhiều parquet transaction chunk
trans_folder = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data"

transaction_chunks = []
for i in range(20):   # vì có chunk_0 đến chunk_19
    path_t = fr"{trans_folder}\sale_pers.purchase_history_daily_chunk_{i}.parquet"
    print("Loading:", path_t)

    df_t = pl.read_parquet(
        path_t,
        columns=["customer_id", "item_id", "created_date"]  # chỉ load cột cần thiết
    )

    transaction_chunks.append(df_t)

print("Tải xong tất cả transaction chunks")


Loading: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_0.parquet
Loading: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_1.parquet
Loading: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_2.parquet
Loading: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_3.parquet
Loading: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_4.parquet
Loading: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_5.parquet
Loading: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_h

In [53]:
df_trans = pl.concat(transaction_chunks, how="vertical")
print("TRANSACTION shape:", df_trans.shape)
print(df_trans.head())


TRANSACTION shape: (35729825, 3)
shape: (5, 3)
┌─────────────┬───────────────┬──────────────┐
│ customer_id ┆ item_id       ┆ created_date │
│ ---         ┆ ---           ┆ ---          │
│ i32         ┆ str           ┆ date         │
╞═════════════╪═══════════════╪══════════════╡
│ 5254214     ┆ 7115000000004 ┆ 2024-12-24   │
│ 7573232     ┆ 0029130000030 ┆ 2024-12-24   │
│ 8187418     ┆ 3496000000053 ┆ 2024-12-24   │
│ 8187418     ┆ 2700000000002 ┆ 2024-12-24   │
│ 6931560     ┆ 0029110000036 ┆ 2024-12-28   │
└─────────────┴───────────────┴──────────────┘


In [54]:
df_join = df_trans.join(
    df_item,
    on="item_id",
    how="left"
)

print("JOIN shape:", df_join.shape)
print(df_join.head())


JOIN shape: (35729825, 5)
shape: (5, 5)
┌─────────────┬───────────────┬──────────────┬──────────────────┬───────────────┐
│ customer_id ┆ item_id       ┆ created_date ┆ category_l1      ┆ price_segment │
│ ---         ┆ ---           ┆ ---          ┆ ---              ┆ ---           │
│ i32         ┆ str           ┆ date         ┆ str              ┆ i64           │
╞═════════════╪═══════════════╪══════════════╪══════════════════╪═══════════════╡
│ 5254214     ┆ 7115000000004 ┆ 2024-12-24   ┆ Thực phẩm cho bé ┆ 0             │
│ 7573232     ┆ 0029130000030 ┆ 2024-12-24   ┆ Thực phẩm cho bé ┆ 0             │
│ 8187418     ┆ 3496000000053 ┆ 2024-12-24   ┆ Thời trang       ┆ 0             │
│ 8187418     ┆ 2700000000002 ┆ 2024-12-24   ┆ Vệ sinh          ┆ 1             │
│ 6931560     ┆ 0029110000036 ┆ 2024-12-28   ┆ Thực phẩm cho bé ┆ 1             │
└─────────────┴───────────────┴──────────────┴──────────────────┴───────────────┘


In [56]:
important_cats = ["Sữa", "Tã"]

df_filtered = df_join.filter(
    pl.col("category_l1").is_in(important_cats)
)

print("Dữ liệu sau khi lọc 3 nhóm quan trọng:", df_filtered.shape)
print(df_filtered.head())


Dữ liệu sau khi lọc 3 nhóm quan trọng: (7661159, 5)
shape: (5, 5)
┌─────────────┬───────────────┬──────────────┬─────────────┬───────────────┐
│ customer_id ┆ item_id       ┆ created_date ┆ category_l1 ┆ price_segment │
│ ---         ┆ ---           ┆ ---          ┆ ---         ┆ ---           │
│ i32         ┆ str           ┆ date         ┆ str         ┆ i64           │
╞═════════════╪═══════════════╪══════════════╪═════════════╪═══════════════╡
│ 3353278     ┆ 2242000910001 ┆ 2024-12-24   ┆ Tã          ┆ 1             │
│ 7573978     ┆ 0020010000438 ┆ 2024-12-24   ┆ Sữa         ┆ 1             │
│ 4810993     ┆ 2263000000021 ┆ 2024-12-28   ┆ Tã          ┆ 0             │
│ 7901749     ┆ 3773000000004 ┆ 2024-12-24   ┆ Sữa         ┆ 2             │
│ 7555248     ┆ 3773000000004 ┆ 2024-12-24   ┆ Sữa         ┆ 2             │
└─────────────┴───────────────┴──────────────┴─────────────┴───────────────┘


In [57]:
customer_segment_count = (
    df_filtered
    .group_by(["customer_id", "price_segment"])
    .agg(pl.count().alias("purchase_count"))
)

print("=== Mẫu thống kê số lần mua theo phân khúc giá ===")
print(customer_segment_count.head(10))


C:\Users\PC\AppData\Local\Temp\ipykernel_7772\3378104036.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("purchase_count"))


=== Mẫu thống kê số lần mua theo phân khúc giá ===
shape: (10, 3)
┌─────────────┬───────────────┬────────────────┐
│ customer_id ┆ price_segment ┆ purchase_count │
│ ---         ┆ ---           ┆ ---            │
│ i32         ┆ i64           ┆ u32            │
╞═════════════╪═══════════════╪════════════════╡
│ 1872217     ┆ 0             ┆ 1              │
│ 6618165     ┆ 0             ┆ 5              │
│ 3523954     ┆ 1             ┆ 2              │
│ 4438097     ┆ 1             ┆ 11             │
│ 3437343     ┆ 1             ┆ 4              │
│ 7466152     ┆ 0             ┆ 1              │
│ 6842213     ┆ 1             ┆ 3              │
│ 4523553     ┆ 0             ┆ 2              │
│ 5016373     ┆ 1             ┆ 2              │
│ 7175948     ┆ 0             ┆ 5              │
└─────────────┴───────────────┴────────────────┘


In [58]:
customer_pivot = (
    customer_segment_count
    .pivot(
        index="customer_id",
        columns="price_segment",
        values="purchase_count",
        aggregate_function="first",
    )
    .fill_null(0)
    .rename({"0": "seg_0", "1": "seg_1", "2": "seg_2"})
)

print("=== Pivot bảng phân khúc giá theo customer ===")
print(customer_pivot.head(10))


C:\Users\PC\AppData\Local\Temp\ipykernel_7772\2920628322.py:3: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(


=== Pivot bảng phân khúc giá theo customer ===
shape: (10, 4)
┌─────────────┬───────┬───────┬───────┐
│ customer_id ┆ seg_0 ┆ seg_1 ┆ seg_2 │
│ ---         ┆ ---   ┆ ---   ┆ ---   │
│ i32         ┆ u32   ┆ u32   ┆ u32   │
╞═════════════╪═══════╪═══════╪═══════╡
│ 1872217     ┆ 1     ┆ 1     ┆ 0     │
│ 6618165     ┆ 5     ┆ 0     ┆ 0     │
│ 3523954     ┆ 0     ┆ 2     ┆ 0     │
│ 4438097     ┆ 1     ┆ 11    ┆ 6     │
│ 3437343     ┆ 0     ┆ 4     ┆ 0     │
│ 7466152     ┆ 1     ┆ 7     ┆ 1     │
│ 6842213     ┆ 3     ┆ 3     ┆ 2     │
│ 4523553     ┆ 2     ┆ 1     ┆ 0     │
│ 5016373     ┆ 1     ┆ 2     ┆ 0     │
│ 7175948     ┆ 5     ┆ 23    ┆ 0     │
└─────────────┴───────┴───────┴───────┘


In [60]:
num_customers = customer_pivot.select(pl.count()).item()
print("Số lượng khách hàng:", num_customers)


Số lượng khách hàng: 1276677


C:\Users\PC\AppData\Local\Temp\ipykernel_7772\1016690670.py:1: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  num_customers = customer_pivot.select(pl.count()).item()


In [61]:
customer_luxury = customer_pivot.with_columns([
    # max segment theo logic: ưu tiên phân khúc cao hơn nếu tie
    pl.when((pl.col("seg_2") >= pl.col("seg_1")) & (pl.col("seg_2") >= pl.col("seg_0")))
        .then(2)
    .when((pl.col("seg_1") >= pl.col("seg_0")) & (pl.col("seg_1") >= pl.col("seg_2")))
        .then(1)
    .otherwise(0)
    .alias("luxury_level")
])

print("=== Mẫu luxury level của customer ===")
print(customer_luxury.head(10))


=== Mẫu luxury level của customer ===
shape: (10, 5)
┌─────────────┬───────┬───────┬───────┬──────────────┐
│ customer_id ┆ seg_0 ┆ seg_1 ┆ seg_2 ┆ luxury_level │
│ ---         ┆ ---   ┆ ---   ┆ ---   ┆ ---          │
│ i32         ┆ u32   ┆ u32   ┆ u32   ┆ i32          │
╞═════════════╪═══════╪═══════╪═══════╪══════════════╡
│ 1872217     ┆ 1     ┆ 1     ┆ 0     ┆ 1            │
│ 6618165     ┆ 5     ┆ 0     ┆ 0     ┆ 0            │
│ 3523954     ┆ 0     ┆ 2     ┆ 0     ┆ 1            │
│ 4438097     ┆ 1     ┆ 11    ┆ 6     ┆ 1            │
│ 3437343     ┆ 0     ┆ 4     ┆ 0     ┆ 1            │
│ 7466152     ┆ 1     ┆ 7     ┆ 1     ┆ 1            │
│ 6842213     ┆ 3     ┆ 3     ┆ 2     ┆ 1            │
│ 4523553     ┆ 2     ┆ 1     ┆ 0     ┆ 0            │
│ 5016373     ┆ 1     ┆ 2     ┆ 0     ┆ 1            │
│ 7175948     ┆ 5     ┆ 23    ┆ 0     ┆ 1            │
└─────────────┴───────┴───────┴───────┴──────────────┘


In [62]:
level_counts = (
    customer_luxury
    .group_by("luxury_level")
    .agg(pl.count().alias("num_customers"))
    .sort("luxury_level")
)

print("\n=== Số lượng khách hàng theo từng luxury_level ===")
print(level_counts)



=== Số lượng khách hàng theo từng luxury_level ===
shape: (3, 2)
┌──────────────┬───────────────┐
│ luxury_level ┆ num_customers │
│ ---          ┆ ---           │
│ i32          ┆ u32           │
╞══════════════╪═══════════════╡
│ 0            ┆ 399426        │
│ 1            ┆ 784154        │
│ 2            ┆ 93097         │
└──────────────┴───────────────┘


C:\Users\PC\AppData\Local\Temp\ipykernel_7772\3369559557.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("num_customers"))


In [63]:
output_path = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\customer_luxury.parquet"

# Tạo thư mục nếu chưa tồn tại
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Lưu file
customer_luxury.write_parquet(output_path)

print("Đã lưu file thành công tại:")
print(output_path)

Đã lưu file thành công tại:
D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\customer_luxury.parquet


### **Đặc trưng:** Tuổi hiện tại của bé
Dựa trên age_group, step_1, step_2 và mom

In [ ]:
import glob
import re
from datetime import date

In [83]:
ITEM_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\sale_pers.item_chunk_0.parquet"
TRANS_DIR =  r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data"

OUTPUT_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\customer_age_features.parquet"

PREDICTION_DATE = date(2025, 1, 1)

STEP1_AGE_MONTHS = 3.0       # Step1 dùng cho 0–6M → trung bình ~3M
AGE_0_3M_MONTHS = 1.5        # 0–3M → trung bình ~1.5M
MOM_AGE_MONTHS = 0.0         # bạn yêu cầu = 0


In [84]:
print("Đang load ITEM...")
item = pl.read_parquet(ITEM_PATH)

item = item.select([
    "item_id", 
    "category_l1", "category_l2",
    "description_final",
    "age_group_final"
])

item = item.with_columns(
    pl.col("description_final").str.to_lowercase().alias("desc_lc")
)



Đang load ITEM...


In [85]:
# FLAG Step1 (chỉ nhận khi category_l1 = Sữa)

step1_positive = [
    "step 1", "step-1", "step1",
    "stage 1", "stage-1", "stage1",
    "cho bé 0+", "cho bé 0-6", "từ 0 tháng",
    "trẻ sơ sinh", "newborn"
]

step1_negative = [
    "bước 1:"       # chắc chắn là hướng dẫn
]


item = item.with_columns([

    # Positive: chứa 1 trong các pattern
    pl.any_horizontal([
        pl.col("desc_lc").str.contains(pat)
        for pat in step1_positive
    ]).alias("tmp_step1_pos"),

    # Negative
    pl.any_horizontal([
        pl.col("desc_lc").str.contains(pat)
        for pat in step1_negative
    ]).alias("tmp_step1_neg"),
])

item = item.with_columns([
    (
        (pl.col("category_l1") == "Sữa") &
        pl.col("tmp_step1_pos") &
        (~pl.col("tmp_step1_neg"))
    ).alias("is_step1")
]).drop(["tmp_step1_pos", "tmp_step1_neg"])



In [86]:
# FLAG is_age_0_3M

#Logic: chỉ cần CHỨA “0” và CHỨA “M”.
item = item.with_columns(
    pl.col("age_group_final").str.to_lowercase().alias("age_lc")
)

item = item.with_columns([
    (
        pl.col("age_lc").str.contains("0") &
        pl.col("age_lc").str.contains("m")
    ).alias("is_age_0_3M")
])


In [87]:
#FLAG is_mom (cẩn thận ĐẦM BẦU)

mom_positive = [
    "sữa bầu",
    "sua bau",
    "sữa cho mẹ",
    "dinh dưỡng cho mẹ",
    "dành cho mẹ",
    "mom"
]


mom_negative = [
    "đầm bầu", "váy bầu", "áo bầu"
]

item = item.with_columns([
    pl.any_horizontal([
        pl.col("desc_lc").str.contains(pat)
        for pat in mom_positive
    ]).alias("tmp_mom_pos"),

    pl.any_horizontal([
        pl.col("desc_lc").str.contains(pat)
        for pat in mom_negative
    ]).alias("tmp_mom_neg"),
])

item = item.with_columns([
    (
        (pl.col("tmp_mom_pos")) & 
        (~pl.col("tmp_mom_neg"))
    ).alias("is_mom")
]).drop(["tmp_mom_pos", "tmp_mom_neg"])


In [88]:
#ĐỌC 20 CHUNK TRANSACTION + JOIN ITEM + TÍNH NGÀY MIN/MAX

transaction_files = sorted(glob.glob(TRANS_DIR + r"\sale_pers.purchase_history_daily_chunk_*.parquet"))
print("Số file transaction:", len(transaction_files))

per_chunk_stats = []


Số file transaction: 20


In [89]:
for file in transaction_files:
    print("Đang xử lý:", file)

    trans = (
        pl.read_parquet(file)
        .select(["item_id", "customer_id", "created_date"])
        .with_columns(pl.col("created_date").cast(pl.Date))
    )

    # JOIN ITEM
    trans_j = trans.join(
        item.select(["item_id", "is_step1", "is_age_0_3M", "is_mom"]),
        on="item_id",
        how="left"
    )

    # AGG theo customer
    stats = (
        trans_j.group_by("customer_id")
        .agg([
            pl.col("created_date").filter(pl.col("is_step1")).min().alias("first_date_buy_step1"),
            pl.col("created_date").filter(pl.col("is_age_0_3M")).min().alias("first_date_buy_age_group_0_3M"),
            pl.col("created_date").filter(pl.col("is_mom")).max().alias("last_date_buy_milk4mom"),
        ])
    )

    per_chunk_stats.append(stats)


Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_0.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_1.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_10.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_11.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_12.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_13.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing 

In [90]:
#GỘP KẾT QUẢ CHUNK + LẤY MIN/MAX CUỐI CÙNG

print("\n=== Gộp tất cả chunk lại ===")
customer_dates_all = pl.concat(per_chunk_stats, how="vertical_relaxed")
print("Shape sau gộp:", customer_dates_all.shape)


customer_dates_final = (
    customer_dates_all
    .group_by("customer_id")
    .agg([
        pl.col("first_date_buy_step1").min().alias("first_date_buy_step1"),
        pl.col("first_date_buy_age_group_0_3M").min().alias("first_date_buy_age_group_0_3M"),
        pl.col("last_date_buy_milk4mom").max().alias("last_date_buy_milk4mom"),
    ])
)



=== Gộp tất cả chunk lại ===
Shape sau gộp: (9875793, 4)


In [92]:
print("\n=== 20 dòng mẫu có thông tin tuổi ===")
sample_nonnull = customer_dates_final.filter(
    pl.col("first_date_buy_step1").is_not_null() |
    pl.col("first_date_buy_age_group_0_3M").is_not_null() |
    pl.col("last_date_buy_milk4mom").is_not_null()
).head(20)

print(sample_nonnull)



=== 20 dòng mẫu có thông tin tuổi ===
shape: (20, 4)
┌─────────────┬──────────────────────┬───────────────────────────────┬────────────────────────┐
│ customer_id ┆ first_date_buy_step1 ┆ first_date_buy_age_group_0_3M ┆ last_date_buy_milk4mom │
│ ---         ┆ ---                  ┆ ---                           ┆ ---                    │
│ i32         ┆ date                 ┆ date                          ┆ date                   │
╞═════════════╪══════════════════════╪═══════════════════════════════╪════════════════════════╡
│ 7602379     ┆ null                 ┆ 2024-06-15                    ┆ 2024-06-15             │
│ 1690435     ┆ null                 ┆ 2024-02-03                    ┆ 2024-02-03             │
│ 7124587     ┆ 2024-01-09           ┆ 2024-01-09                    ┆ null                   │
│ 5216879     ┆ null                 ┆ 2024-02-13                    ┆ null                   │
│ 4831490     ┆ 2024-10-27           ┆ 2024-03-11                    ┆ 2024-06-04 

In [94]:
# TÍNH TUỔI HIỆN TẠI (age_by_*)
pred_date_lit = pl.lit(PREDICTION_DATE).cast(pl.Date)

customer_with_age = (
    customer_dates_final
    .with_columns([
        # Lấy số ngày từ hiệu hai ngày
        (pred_date_lit - pl.col("first_date_buy_step1")).dt.total_days().alias("days_from_step1"),
        (pred_date_lit - pl.col("first_date_buy_age_group_0_3M")).dt.total_days().alias("days_from_age_group"),
        (pred_date_lit - pl.col("last_date_buy_milk4mom")).dt.total_days().alias("days_from_milk4mom"),
    ])
    .with_columns([
        (pl.col("days_from_step1") / 30).alias("months_from_step1"),
        (pl.col("days_from_age_group") / 30).alias("months_from_age_group"),
        (pl.col("days_from_milk4mom") / 30).alias("months_from_milk4mom"),
    ])
    .with_columns([
        # FINAL AGE
        (STEP1_AGE_MONTHS + pl.col("months_from_step1")).alias("age_by_step1"),
        (AGE_0_3M_MONTHS + pl.col("months_from_age_group")).alias("age_by_age_group"),
        (MOM_AGE_MONTHS + pl.col("months_from_milk4mom")).alias("age_by_milk4mom"),
    ])
    .select([
        "customer_id",
        "first_date_buy_step1", "age_by_step1",
        "first_date_buy_age_group_0_3M", "age_by_age_group",
        "last_date_buy_milk4mom", "age_by_milk4mom",
    ])
)


In [95]:
# check

print("\n=== THỐNG KÊ NON-NULL ===")

print("Có Step1:", customer_with_age.filter(pl.col("age_by_step1").is_not_null()).height)
print("Có age_group 0–3M:", customer_with_age.filter(pl.col("age_by_age_group").is_not_null()).height)
print("Có Mom:", customer_with_age.filter(pl.col("age_by_milk4mom").is_not_null()).height)



=== THỐNG KÊ NON-NULL ===
Có Step1: 318575
Có age_group 0–3M: 1395373
Có Mom: 482929


In [97]:
# check

print("\n=== Mẫu khách có cả 3 tín hiệu ===")
print(
    customer_with_age
    .filter(
        pl.col("age_by_step1").is_not_null() &
        pl.col("age_by_age_group").is_not_null() &
        pl.col("age_by_milk4mom").is_not_null()
    )
    .head(20)
)



=== Mẫu khách có cả 3 tín hiệu ===
shape: (20, 7)
┌─────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ customer_id ┆ first_date_b ┆ age_by_step ┆ first_date_ ┆ age_by_age_ ┆ last_date_b ┆ age_by_milk │
│ ---         ┆ uy_step1     ┆ 1           ┆ buy_age_gro ┆ group       ┆ uy_milk4mom ┆ 4mom        │
│ i32         ┆ ---          ┆ ---         ┆ up_0_3M     ┆ ---         ┆ ---         ┆ ---         │
│             ┆ date         ┆ f64         ┆ ---         ┆ f64         ┆ date        ┆ f64         │
│             ┆              ┆             ┆ date        ┆             ┆             ┆             │
╞═════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 4831490     ┆ 2024-10-27   ┆ 5.2         ┆ 2024-03-11  ┆ 11.366667   ┆ 2024-06-04  ┆ 7.033333    │
│ 5922091     ┆ 2024-03-19   ┆ 12.6        ┆ 2024-03-19  ┆ 11.1        ┆ 2024-07-28  ┆ 5.233333    │
│ 5448077     ┆ 2024-01-12   ┆ 14.833333

In [98]:
customer_with_age.write_parquet(OUTPUT_PATH)
print("Đã lưu:", OUTPUT_PATH)


Đã lưu: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\customer_age_features.parquet


### **Đặc trưng:** Có mua đa dạng brand không. 
Cũng chỉ xét "Sữa", "Tã" và "Sữa nước"

### **Đặc trưng về hành vi mua hàng:** Xét hành vi mua hàng dựa trên độ tuổi của bé
Lấy độ đa dạng trong category_1, lấy top K sản phẩm tại mỗi loại để xem người dùng có bé trong độ tuổi đó thì thường mua những sản phẩm nào

### **Đặc trưng về hành vi mua hàng:** Thống kê những sản phẩm hay mua chung và số lần mua chung: item 1 | item 2 | #cooc (Cũng xét theo ngày)

### Discount user: Người dùng đó có mua hàng discount hay k. Tính dựa trên số lần mua trong tháng
Cũng có thể sẽ là gom cụm: mua ít, mua vừa, mua nhiều

### Đếm số lượng mặt hàng được bán ra (Mặt hàng được bán ra càng nhiều thì khả năng người ta mua hàng sẽ càng cao )  - **Có thể lấy top 100**
(Hệ số phổ biến của sản phẩm)

### Lần cuối cùng mua hàng của khách hàng